In [11]:
import random
import json
from datetime import datetime, timedelta
from util.json_handler import JsonHandler
from util.string_parser import StringParser
import fun.FlaskFunctions as ff

In [96]:
settings = JsonHandler('data/settings.json')

In [13]:
media_folders = [ ff.linuxify_path(f) for f in settings.getValue("media_folders") ]
filename_formats = settings.getValue("filename_formats")

In [15]:
# load files
media = {}
for f in media_folders:
    print(f)
    media[f] = ff.get_media_from_dirs([f])

/mnt/a/Whispera/gifsCollection
Scanning folder (1/1) "/mnt/a/Whispera/gifsCollection"  ... found 1_792 media
/mnt/a/Whispera/pics/BobbiStarrPics
Scanning folder (1/1) "/mnt/a/Whispera/pics/BobbiStarrPics"  ... found 77_447 media
/mnt/a/Whispera/pics/Erotic Illustration Collections
Scanning folder (1/1) "/mnt/a/Whispera/pics/Erotic Illustration Collections"  ... found 28_365 media
/mnt/a/Whispera/pics/Sasha Grey 1000 facials pic sets
Scanning folder (1/1) "/mnt/a/Whispera/pics/Sasha Grey 1000 facials pic sets"  ... found 630 media


In [65]:
def path_components(path):
    from pathlib import Path
    obj = Path(path)
    stem = obj.stem
    suffix = obj.suffix
    parents = [ p for p in str(obj.parent).split('/') if p != '' ]
    return parents, stem, suffix

In [94]:
from typing import Any

def parse_data_from_path(stem: str, parser: StringParser) -> dict[str, Any]:
    data = parser.parse(stem)
    if data == None:
        return {}
    if 'source_id_TwitterSeleniumScraped' in data:
        data['source_id'] = data['source_id_TwitterSeleniumScraped'].replace(' photo ', '-')
    if 'date_uploaded_dt' in data:
        data['date_uploaded'] = data['date_uploaded_dt'].split('T')[0]
    data['tags'] = data.get('tags', []) + data['title'].split()
    return data

In [87]:
# 
filename_formats = [
    "{creator} - {source} - [{date_uploaded}];opt {title} [{source_id}];opt",
    "{creator} [{date_uploaded_full}] [{source_id_TwitterSeleniumScraped}] {title}"
]
parser = StringParser(filename_formats)

In [95]:
# 
random.seed(6)
for i in range(10):
    f = random.choice(media_folders)
    media_path = random.choice(media.get(f, []))
    print('\n({}) "{}"'.format(i+1, media_path))
    parents, stem, suffix = path_components(media_path)
    
    print('    parents:   "{}"'.format(parents))
    print('    stem:      "{}"'.format(stem))
    print('    suffix:    "{}"'.format(suffix))
    
    data = parse_data_from_path(stem, parser)
    if data:
        for k, v in data.items():
            print('   {:<20}: {}'.format(k, v))
    else:
        print('  UNKNOWN')


(1) "Reddit/throatpussy/throatpussy - Reddit - [2024-11-17] Nothing Better Than Balcony Throat Pussy  #1gtdaqk.mp4"
    parents:   "['Reddit', 'throatpussy']"
    stem:      "throatpussy - Reddit - [2024-11-17] Nothing Better Than Balcony Throat Pussy  #1gtdaqk"
    suffix:    ".mp4"
   creator             : throatpussy
   source              : Reddit
   date_uploaded       : 2024-11-17
   title               : Nothing Better Than Balcony Throat Pussy 
   tags                : ['1gtdaqk', 'Nothing', 'Better', 'Than', 'Balcony', 'Throat', 'Pussy']

(2) "Aelion_Draws/Aelion_Draws [2024-02-24T21-53-38] [1761509718386999544 photo 1] First I do grayscale then I go to color and render it from there. ))Flat vs Render (kinda).jpg"
    parents:   "['Aelion_Draws']"
    stem:      "Aelion_Draws [2024-02-24T21-53-38] [1761509718386999544 photo 1] First I do grayscale then I go to color and render it from there. ))Flat vs Render (kinda)"
    suffix:    ".jpg"
   creator             : Aelion_Draws

In [104]:
from datetime import datetime
datetime.now().strftime('%Y-%m-%dT%H-%M-%S')

'2025-01-06T09-46-06'